# Session 3 — Your turn: Training a small GPT

Two small pieces of the training and sampling code, rewritten and tested on their own: the learning-rate schedule and the top-p filter.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import math
import numpy as np
from aij_check import check, close, summary

## Task 1: Warm-up and cosine decay

Write `lr_at(step, total_steps, peak, warmup, floor=0.1)`: for the first `warmup` steps the rate rises linearly, `peak * (step + 1) / warmup`; afterwards it follows a cosine curve from `peak` down to `floor * peak`
at the last step: `peak * (floor + (1 - floor) * 0.5 * (1 + cos(pi * (step - warmup) / (total_steps - warmup))))`.

In [ ]:
def lr_at(step, total_steps, peak, warmup, floor=0.1):
    ...

In [ ]:
check("the first step is peak / warmup", lambda: close(lr_at(0, 1000, 3e-3, 100), 3e-5, 1e-12))
check("the last warm-up step reaches the peak", lambda: close(lr_at(99, 1000, 3e-3, 100), 3e-3, 1e-12))
check("just after warm-up it is at the peak", lambda: close(lr_at(100, 1000, 3e-3, 100), 3e-3, 1e-12))
check("halfway through the decay it is midway between peak and floor", lambda: close(lr_at(550, 1000, 3e-3, 100), 3e-3 * (0.1 + 0.9 * 0.5), 1e-9))
check("after warm-up it never falls below the floor", lambda: min(lr_at(s, 1000, 3e-3, 100) for s in range(100, 1000)) >= 3e-4 - 1e-12)
summary()

## Task 2: Top-p filtering

Write `top_p_filter(probs, p)`: sort the probabilities, keep the smallest set of tokens whose probabilities add up to **at least** `p` (always at least one token), set the others to 0 and renormalise so the result sums to 1. Return an array in the original order.

In [ ]:
def top_p_filter(probs, p):
    ...


demo = np.array([0.50, 0.20, 0.15, 0.10, 0.05])

In [ ]:
check("p = 0.9 keeps four of the five tokens", lambda: int((top_p_filter(demo, 0.9) > 0).sum()) == 4)
check("the kept probabilities are renormalised", lambda: close(top_p_filter(demo, 0.9).sum(), 1.0, 1e-9) and close(top_p_filter(demo, 0.9)[0], 0.5 / 0.95, 1e-9))
check("a tiny p keeps only the most likely token", lambda: close(top_p_filter(demo, 0.1), [1, 0, 0, 0, 0], 1e-9))
check("p = 1 keeps everything", lambda: close(top_p_filter(demo, 1.0), demo, 1e-9))
check("it works when the input is not sorted", lambda: close(top_p_filter(demo[::-1], 0.7), top_p_filter(demo, 0.7)[::-1], 1e-9))
summary()